# MODELO PREDICTIVO

Lo primero es juntar las 4 tablas pivotadas:

1. Datos demogarficos
2. Comorbilidades
3. Medicación
4. Pruebas Médicas

In [ ]:
import pandas as pd
df_demografico=pd.read_excel(r"DATA_ROOT\Limpieza BBDD\Datos demograficos\datos_demograficos_diabeticos.xlsx")
df_comorbilidades=pd.read_excel(r"DATA_ROOT\Limpieza BBDD\Comorbilidades\Comorbilidades.xlsx")
df_med=pd.read_excel(r"DATA_ROOT\Limpieza BBDD\Medicacion\dosis_med_pacientes.xlsx")
df_lab=pd.read_excel(r"DATA_ROOT\Limpieza BBDD\Pruebas medicas\df_labevents_imputado_custom_vf.xlsx")


In [ ]:
df_demografico.head()

In [ ]:
df_demografico.shape[0]

In [ ]:
df_comorbilidades.head()

In [ ]:
df_comorbilidades=df_comorbilidades.drop(['Unnamed: 0','subject_id', 'hadm_id','admittime','gender','anchor_age','dod'], axis=1)
df_comorbilidades.head()

In [ ]:
df_comorbilidades.shape[0]

In [ ]:
df_comorbilidades_pivotada=pd.pivot_table(df_comorbilidades, 
                          index='patient_id', 
                          columns='CATEGORÍA', 
                          aggfunc='size', 
                          fill_value=0)
# Renombrar las columnas para que contengan 1 si la enfermedad está presente
df_comorbilidades_pivotada[df_comorbilidades_pivotada > 0] = 1

In [ ]:
df_comorbilidades_pivotada.shape[0]

In [ ]:
df_comorbilidades_pivotada.head()

En el caso de comorbilidades, 0=no tiene la enfermedad y 1=tiene la enfermedad

In [ ]:
df_med.head()

In [ ]:
med_pat_list=list(df_med['patient_id'].unique())
med_pat_list

In [ ]:
df_med.shape[0]

In [ ]:
df_med.columns

In [ ]:
df_lab.head()

In [ ]:
df_lab=df_lab.drop(['subject_id', 'hadm_id','charttime'], axis=1)

In [ ]:
df_lab.rename(columns={'identificador': 'patient_id'}, inplace=True)

In [ ]:
df_lab.head()

In [ ]:
#Dividimos entre 10 los resultados de las pruebas médicas 
df_lab.loc[:, df_lab.columns != 'patient_id'] = df_lab.loc[:, df_lab.columns != 'patient_id'] / 10


In [ ]:
df_lab.head()

In [ ]:
df_lab.shape[0]

In [ ]:
df_lab.columns

Ahora que todas las tablas tienen el mismo formato, vamos a unirlas todas por el patient id

In [ ]:
# Unir los DataFrames por 'patient_id'
df_merged = df_demografico.merge(df_comorbilidades_pivotada, on='patient_id', how='outer') \
               .merge(df_med, on='patient_id', how='outer') \
               .merge(df_lab, on='patient_id', how='outer')

In [ ]:
df_merged.head()

In [ ]:
df_merged.shape[0]

## ANALISIS EXPLORATORIO (EDA)

### 1- Revisión de los Datos- Data Profiling

#### Tipo de datos

In [ ]:
df_merged.info(verbose=True, show_counts=True)

In [ ]:
# Obtener los tipos de datos de las columnas
data_types = df_merged.dtypes

# Clasificar las columnas por tipo de dato
types_dict = {}
for col, dtype in data_types.items():
    if dtype not in types_dict:
        types_dict[dtype] = [col]
    else:
        types_dict[dtype].append(col)

# Mostrar las columnas clasificadas por tipo de dato
for dtype, columns in types_dict.items():
    print(f"{dtype}: {columns}")


Cambiamos el tipo de datos

In [ ]:
# Verificar que solo tiene valores binarios
binary_vars = [
    'Cardiología', 'Cirugía / Procedimientos Médicos', 'Enfermedades Infecciosas', 
    'Factores Sociales / Exposición / Historial', 'Gastroenterología / Digestivo', 
    'Insuficiencia Renal', 'Metabólicas Y Nutricionales', 'Nefrología', 
    'Neumología', 'Oncología', 'Reumatología', 'Salud Mental Y Neurológica'
]

# Comprobar los valores únicos de cada una de las variables binarias
for var in binary_vars:
    print(f"Valores únicos para {var}: {df_merged[var].unique()}")

In [ ]:
# Convertir las variables binarias a tipo bool
df_merged[binary_vars] = df_merged[binary_vars].astype('bool')

# Cambiar el tipo de datos de 'patient_id' y 'gender' a 'category'
df_merged['patient_id'] = df_merged['patient_id'].astype('category')
df_merged['gender'] = df_merged['gender'].astype('category')



In [ ]:
# Obtener los tipos de datos de las columnas
data_types = df_merged.dtypes

# Clasificar las columnas por tipo de dato
types_dict = {}
for col, dtype in data_types.items():
    if dtype not in types_dict:
        types_dict[dtype] = [col]
    else:
        types_dict[dtype].append(col)

# Mostrar las columnas clasificadas por tipo de dato
for dtype, columns in types_dict.items():
    print(f"{dtype}: {columns}")


#### Estadísticos descriptivos de las variables numéricas

In [ ]:
df_merged_describe=df_merged.describe().round(2)
df_merged_describe

In [ ]:
# Identificar las variables con mínimo 0 y máximo 1
binary_vars = df_merged_describe.columns[(df_merged_describe.loc['min'] == 0.0) & (df_merged_describe.loc['max'] == 1.0)].to_list()
binary_vars

Estas variables tienen que clasificarse como tipo binario:

In [ ]:
# Convertir estas variables a tipo binario (int64)
df_merged[binary_vars] = df_merged[binary_vars].astype('bool')

In [ ]:
# Obtener los tipos de datos de las columnas
data_types = df_merged.dtypes

# Clasificar las columnas por tipo de dato
types_dict = {}
for col, dtype in data_types.items():
    if dtype not in types_dict:
        types_dict[dtype] = [col]
    else:
        types_dict[dtype].append(col)

# Mostrar las columnas clasificadas por tipo de dato
for dtype, columns in types_dict.items():
    print(f"{dtype}: {columns}")


In [ ]:
df_merged_describe=df_merged.describe().round(2)
df_merged_describe

In [ ]:
#Identificar valores constantes
cte_vars=df_merged_describe.columns[(df_merged_describe.loc['min'] == df_merged_describe.loc['max'])].to_list()
cte_vars

Estas variables son constantes, lo que significa que su varianza es 0 por lo que no van a ser muy informativas, asi que son potentes candidatas de eliminarse

In [ ]:
# Asegúrate de que el índice es de tipo string
df_merged.index = df_merged.index.map(str)

# Ahora exporta normalmente
#df_merged.to_excel("tabla_completa_final.xlsx", index=True)


### Definimos las variables y el target

In [ ]:
target='Insuficiencia Renal'
features_list = list(df_merged.columns)
features_list.remove(target)
features_list.remove('patient_id')
print(features_list)

### 2- Visualización

### 2.1. Análisis Univariante

#### Distribucion de la variable target

In [ ]:
target='Insuficiencia Renal'
df_merged[target].info()

In [ ]:
import seaborn as sns 
import matplotlib.pyplot as plt

# Crear el gráfico
ax = sns.countplot(x=target, data=df_merged, palette=['#3f3faf', '#5481a6'])

# Títulos y etiquetas
plt.title('Distribución de la Variable Target', fontsize=12)
plt.xlabel('Valor', fontsize=10)
plt.ylabel('Frecuencia', fontsize=10)

#Añadir etiquetas en cada barra
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{height}', 
                (p.get_x() + p.get_width() / 2., height), 
                ha='center', va='bottom', fontsize=9)

# Mostrar el gráfico
plt.show()


#### Distribucion de las variables numéricas

In [ ]:
import matplotlib.pyplot as plt

# Seleccionar solo las variables numéricas
numeric_vars = df_merged.select_dtypes(include=['float64', 'int64'])

# Crear histogramas para cada variable numérica
numeric_vars.hist(bins=20, figsize=(20, 18), grid=True, edgecolor='black')
plt.tight_layout()
plt.show()


Tras realizar un análisis exploratorio de las distribuciones de las variables numéricas en el conjunto de datos, se identificaron algunas observaciones clave que requieren una atención particular:

1. Variables de Medicación:

Se observó que varias variables relacionadas con la medicación, como 'Antiasthmatic', 'Antidiarrheal', 'Muscle Relaxant', 'Sedatives & Analgesics', 'Thrombolytic', 'Topical', 'Topical Analgesic', y 'Topical Antibiotic', presentan una distribución constante, es decir, con una varianza de 0. Esto indica que todos los pacientes que toman estas medicaciones tienen los mismos valores, lo que puede deberse a que los medicamentos son administrados a la misma dosis a todos los pacientes o a la ausencia de variabilidad en los datos. Estas variables requieren una revisión más detallada para comprender su falta de variabilidad, lo que podría ser relevante para el análisis posterior.

En general, las variables correspondientes a medicación no siguen una distribución centralizada (normal o simétrica), lo que sugiere que la mayoría de las dosis de medicación están concentradas en ciertos valores y que hay pocos pacientes con dosis extremadamente altas o bajas. Esto puede ser esperado en ciertos tratamientos médicos, donde las dosis son determinadas por protocolos estándar.

2. Variables de Pruebas de Laboratorio:

En cambio, las variables asociadas a las pruebas de laboratorio, como las concentraciones de glucosa y creatinina, hemoglobina y hematocrito, muestran distribuciones más simétricas o normales. Sin embargo, algunos de estos resultados presentan asimetría en las distribuciones. Esta asimetría puede deberse a la presencia de valores atípicos o a la influencia de factores biológicos en los resultados de los pacientes, como la presencia de comorbilidades o tratamientos médicos que afectan los resultados de laboratorio. Esta asimetría debe tenerse en cuenta al aplicar técnicas estadísticas, ya que puede ser necesario transformar o ajustar los datos para normalizarlos o hacerlos más comparables entre los pacientes.

### Pruebas de laboratorio

#### Detección de outliers

In [ ]:

lab_columns=['Anion Gap-Blood-Chemistry',
       'Bicarbonate-Blood-Chemistry', 'Calcium, Total-Blood-Chemistry',
       'Chloride-Blood-Chemistry', 'Creatinine-Blood-Chemistry',
       'Glucose-Blood-Chemistry', 'Hematocrit-Blood-Hematology',
       'Hemoglobin-Blood-Hematology', 'MCH-Blood-Hematology',
       'MCHC-Blood-Hematology', 'MCV-Blood-Hematology',
       'Magnesium-Blood-Chemistry', 'Phosphate-Blood-Chemistry',
       'Platelet Count-Blood-Hematology', 'Potassium-Blood-Chemistry',
       'RDW-Blood-Hematology', 'Red Blood Cells-Blood-Hematology',
       'Sodium-Blood-Chemistry', 'Urea Nitrogen-Blood-Chemistry',
       'White Blood Cells-Blood-Hematology']
# Definir la función para detectar outliers usando el IQR
def detectar_outliers_iqr(df, columns):
    outliers = {}
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        # Definir los límites inferior y superior
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Identificar los outliers
        outliers[col] = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
    return outliers

# Llamar a la función para detectar los outliers en las columnas de laboratorio
outliers_detectados = detectar_outliers_iqr(df_merged, lab_columns)

# Mostrar los outliers encontrados en cada columna
for col, data in outliers_detectados.items():
    print(f"Outliers en {col}:")
    print(data[[col]])
    print("\n")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Crear una figura con subgráficos
n_cols = 5  # Número de columnas en el gráfico
n_rows = (len(lab_columns) // n_cols) + (len(lab_columns) % n_cols > 0)  # Número de filas necesarias

# Crear subgráficos para los boxplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()  # Convertir la matriz de subgráficos en un arreglo 1D

# Crear un boxplot para cada columna de laboratorio
for i, col in enumerate(lab_columns):
    sns.boxplot(data=df_merged, x=col, ax=axes[i])
    axes[i].set_title(f'Outliers en {col}', fontsize=10)
    axes[i].set_xlabel(col, fontsize=8)
    axes[i].set_ylabel('Valor', fontsize=8)
    axes[i].grid(True)

# Eliminar cualquier subgráfico vacío si el número de columnas no es múltiplo de 5
for j in range(len(lab_columns), len(axes)):
    fig.delaxes(axes[j])

# Ajustar el espacio entre los subgráficos
plt.tight_layout()

# Mostrar los gráficos
plt.show()


- **Presencia de outliers**:

En todas las variables, como se esperaba con la detección de outliers mediante el IQR, hay valores que están fuera de los "bigotes" de los boxplots. Estos puntos están marcados como círculos (puntos individuales fuera del rango intercuartílico), lo que indica que son outliers.

En algunas variables como Glucose-Blood-Chemistry, Potassium-Blood-Chemistry, Sodium-Blood-Chemistry, y Platelet Count-Blood-Hematology, los outliers están más concentrados hacia los valores más altos. Esto puede ser indicativo de pacientes con niveles anormales de estos elementos en la sangre, lo cual puede requerir revisión clínica.

En otras variables, como Urea Nitrogen-Blood-Chemistry, Red Blood Cells-Blood-Hematology, y Phosphate-Blood-Chemistry, los outliers parecen estar más distribuidos, lo que podría sugerir una variabilidad más natural o frecuente en esos parámetros, pero aún fuera de los rangos esperados.

- **Forma de la distribución**:

Algunas variables como Hemoglobin-Blood-Hematology y Hematocrit-Blood-Hematology tienen una distribución en la que los valores atípicos se encuentran relativamente cerca de los valores centrales. Esto sugiere que la mayoría de los pacientes tienen niveles bastante estables, pero algunas mediciones son mucho más altas que el resto.

Variables como MCV-Blood-Hematology y RDW-Blood-Hematology tienen outliers que se alejan bastante del rango central, lo que puede indicar que ciertos pacientes presentan mediciones muy alejadas de la media, lo que podría ser relevante en diagnóstico de anemia u otras condiciones hematológicas.

- **Importancia de los outliers**:

En algunos casos, como Urea Nitrogen-Blood-Chemistry o Creatinine-Blood-Chemistry, los outliers podrían estar reflejando una insuficiencia renal o daño renal, que son condiciones de alta relevancia clínica.


- **Densidad de los datos**:

En general, los boxplots muestran que para muchas variables los datos están concentrados en un rango estrecho (aunque hay outliers). Las distribuciones no son simétricas, lo que indica que algunas variables podrían estar sesgadas hacia la derecha o la izquierda.

In [ ]:
niveles_anormales=['Glucose-Blood-Chemistry', 'Potassium-Blood-Chemistry', 'Sodium-Blood-Chemistry', 'Platelet Count-Blood-Hematology']

# Crear una figura con subgráficos
n_cols = 4  # Número de columnas en el gráfico
n_rows = 1
# Crear subgráficos para los boxplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()  # Convertir la matriz de subgráficos en un arreglo 1D

# Crear un boxplot para cada columna de laboratorio
for i, col in enumerate(niveles_anormales):
    sns.boxplot(data=df_merged, x=col, ax=axes[i])
    axes[i].set_title(f'Outliers en {col}', fontsize=10)
    axes[i].set_xlabel(col, fontsize=8)
    axes[i].set_ylabel('Valor', fontsize=8)
    axes[i].grid(True)

# Eliminar cualquier subgráfico vacío si el número de columnas no es múltiplo de 5
for j in range(len(niveles_anormales), len(axes)):
    fig.delaxes(axes[j])

# Ajustar el espacio entre los subgráficos
plt.tight_layout()

# Mostrar los gráficos
plt.show()


In [ ]:
indicadores_IR=['Urea Nitrogen-Blood-Chemistry','Creatinine-Blood-Chemistry']
# Crear una figura con subgráficos
n_cols = 3  # Número de columnas en el gráfico
n_rows = 1
# Crear subgráficos para los boxplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()  # Convertir la matriz de subgráficos en un arreglo 1D

# Crear un boxplot para cada columna de laboratorio
for i, col in enumerate(indicadores_IR):
    sns.boxplot(data=df_merged, x=col, ax=axes[i])
    axes[i].set_title(f'Outliers en {col}', fontsize=10)
    axes[i].set_xlabel(col, fontsize=8)
    axes[i].set_ylabel('Valor', fontsize=8)
    axes[i].grid(True)

# Eliminar cualquier subgráfico vacío si el número de columnas no es múltiplo de 5
for j in range(len(indicadores_IR), len(axes)):
    fig.delaxes(axes[j])

# Ajustar el espacio entre los subgráficos
plt.tight_layout()

# Mostrar los gráficos
plt.show()


In [ ]:
indicadores_anemia=['MCV-Blood-Hematology','MCH-Blood-Hematology', 'MCHC-Blood-Hematology','Hematocrit-Blood-Hematology','Hemoglobin-Blood-Hematology','RDW-Blood-Hematology', 'Red Blood Cells-Blood-Hematology']
# Crear una figura con subgráficos
n_cols = 4  # Número de columnas en el gráfico
n_rows = 2
# Crear subgráficos para los boxplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()  # Convertir la matriz de subgráficos en un arreglo 1D

# Crear un boxplot para cada columna de laboratorio
for i, col in enumerate(indicadores_anemia):
    sns.boxplot(data=df_merged, x=col, ax=axes[i])
    axes[i].set_title(f'Outliers en {col}', fontsize=10)
    axes[i].set_xlabel(col, fontsize=8)
    axes[i].set_ylabel('Valor', fontsize=8)
    axes[i].grid(True)

# Eliminar cualquier subgráfico vacío si el número de columnas no es múltiplo de 5
for j in range(len(indicadores_anemia), len(axes)):
    fig.delaxes(axes[j])

# Ajustar el espacio entre los subgráficos
plt.tight_layout()

# Mostrar los gráficos
plt.show()




In [ ]:
electrolitos=['Calcium, Total-Blood-Chemistry',
       'Chloride-Blood-Chemistry','Magnesium-Blood-Chemistry', 'Phosphate-Blood-Chemistry',
       'Potassium-Blood-Chemistry',
       'Sodium-Blood-Chemistry']
# Crear una figura con subgráficos
n_cols = 3  # Número de columnas en el gráfico
n_rows = 3
# Crear subgráficos para los boxplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()  # Convertir la matriz de subgráficos en un arreglo 1D

# Crear un boxplot para cada columna de laboratorio
for i, col in enumerate(electrolitos):
    sns.boxplot(data=df_merged, x=col, ax=axes[i])
    axes[i].set_title(f'Outliers en {col}', fontsize=10)
    axes[i].set_xlabel(col, fontsize=8)
    axes[i].set_ylabel('Valor', fontsize=8)
    axes[i].grid(True)

# Eliminar cualquier subgráfico vacío si el número de columnas no es múltiplo de 5
for j in range(len(electrolitos), len(axes)):
    fig.delaxes(axes[j])

# Ajustar el espacio entre los subgráficos
plt.tight_layout()

# Mostrar los gráficos
plt.show()

Para tener una mejor representación de la densidad de la población, se realizaron los siguientes violin plots:

In [ ]:
df_renal=df_merged[df_merged[target]==True]
df_no_renal=df_merged[df_merged[target]==False]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Definir las columnas de las pruebas de laboratorio
lab_columns = ['Anion Gap-Blood-Chemistry', 'Bicarbonate-Blood-Chemistry', 'Calcium, Total-Blood-Chemistry',
               'Chloride-Blood-Chemistry', 'Creatinine-Blood-Chemistry', 'Glucose-Blood-Chemistry', 
               'Hematocrit-Blood-Hematology', 'Hemoglobin-Blood-Hematology', 'MCH-Blood-Hematology',
               'MCHC-Blood-Hematology', 'MCV-Blood-Hematology', 'Magnesium-Blood-Chemistry', 
               'Phosphate-Blood-Chemistry', 'Platelet Count-Blood-Hematology', 'Potassium-Blood-Chemistry',
               'RDW-Blood-Hematology', 'Red Blood Cells-Blood-Hematology', 'Sodium-Blood-Chemistry',
               'Urea Nitrogen-Blood-Chemistry', 'White Blood Cells-Blood-Hematology']


# Crear la columna de grupo si no existe
df_merged['Grupo'] = df_merged[target].map({True: 'Insuficiencia Renal', False: 'Sin Insuficiencia Renal'})

# Crear subplots para visualizar los violin plots
n_cols = 5
n_rows = len(lab_columns) // n_cols + (len(lab_columns) % n_cols > 0)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

# Crear los violin plots
for i, lab_test in enumerate(lab_columns):
    sns.violinplot(
        x='Grupo', y=lab_test, data=df_merged, ax=axes[i], inner="box",
        order=['Insuficiencia Renal', 'Sin Insuficiencia Renal'],
        palette=['#5481a6', '#3f3faf']
    )
    axes[i].set_title(f'Distribución de {lab_test}', fontsize=12)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Valor')

# Eliminar ejes vacíos si sobran
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Ajustar espacio y guardar figura
plt.tight_layout()
plt.savefig('violin_plot_insuficiencia_renal.png')
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Diccionario de nombres legibles
nombres_legibles = {
    'Calcium, Total-Blood-Chemistry': 'Calcio',
    'Chloride-Blood-Chemistry': 'Cloruro',
    'Phosphate-Blood-Chemistry': 'Fosfato',
    'Potassium-Blood-Chemistry': 'Potasio',
    'Sodium-Blood-Chemistry': 'Sodio'
}

# Unidades por prueba
unidades_lab = {
    'Calcium, Total-Blood-Chemistry': 'mg/dL',
    'Chloride-Blood-Chemistry': 'mEq/L',
    'Phosphate-Blood-Chemistry': 'mg/dL',
    'Potassium-Blood-Chemistry': 'mEq/L',
    'Sodium-Blood-Chemistry': 'mEq/L'
}

# Columnas a graficar
lab_columns = list(nombres_legibles.keys())

# Asigna el grupo según la variable objetivo (asegúrate de definir `target`)
df_merged['Grupo'] = df_merged[target].map({True: 'Insuficiencia Renal', False: 'Sin Insuficiencia Renal'})

# Crear subplots
n_cols = 5
n_rows = len(lab_columns) // n_cols + (len(lab_columns) % n_cols > 0)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

# Graficar
for i, lab_test in enumerate(lab_columns):
    ax = axes[i]
    sns.violinplot(
        x='Grupo', y=lab_test, data=df_merged, ax=ax, inner="box",
        order=['Insuficiencia Renal', 'Sin Insuficiencia Renal'],
        palette=['#5481a6', '#3f3faf']
    )

    nombre_legible = nombres_legibles.get(lab_test, lab_test)
    unidad = unidades_lab.get(lab_test, '')
    
    ax.set_title(f'Distribución de {nombre_legible}', fontsize=12)
    ax.set_xlabel('')
    ax.set_ylabel(f'{nombre_legible} ({unidad})')

# Eliminar ejes vacíos
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Título general
fig.suptitle('Distribución de los principales electrolitos de pacientes DT1 con y sin IR', fontsize=16, y=1.02)

# Mostrar
plt.tight_layout()
plt.savefig('violin_plot_electrolitos_DT1_IR_sin_medianas.png', bbox_inches='tight')
plt.show()



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Diccionario de nombres legibles
nombres_legibles = {
    'Glucose-Blood-Chemistry': 'Glucosa',
    'Calcium, Total-Blood-Chemistry': 'Calcio',
    'Chloride-Blood-Chemistry': 'Cloruro',
    'Creatinine-Blood-Chemistry': 'Creatinina',
    'Phosphate-Blood-Chemistry': 'Fosfato',
    'Potassium-Blood-Chemistry': 'Potasio',
    'Sodium-Blood-Chemistry': 'Sodio'
}

# Unidades por prueba
unidades_lab = {
    'Glucose-Blood-Chemistry': 'mg/dL',
    'Calcium, Total-Blood-Chemistry': 'mg/dL',
    'Chloride-Blood-Chemistry': 'mEq/L',
    'Creatinine-Blood-Chemistry': 'mg/dL',
    'Phosphate-Blood-Chemistry': 'mg/dL',
    'Potassium-Blood-Chemistry': 'mEq/L',
    'Sodium-Blood-Chemistry': 'mEq/L'
}

# Rangos normales por prueba
rangos_lab = {
    'Glucose-Blood-Chemistry': (70.0, 100.0),
    'Calcium, Total-Blood-Chemistry': (8.4, 10.3),
    'Chloride-Blood-Chemistry': (96.0, 108.0),
    'Creatinine-Blood-Chemistry': (0.5, 1.2),
    'Phosphate-Blood-Chemistry': (2.7, 4.5),
    'Potassium-Blood-Chemistry': (3.3, 5.1),
    'Sodium-Blood-Chemistry': (133.0, 145.0)
}

# Columnas a graficar
lab_columns = list(nombres_legibles.keys())

# Asegúrate de definir el nombre correcto de tu variable binaria
# Ejemplo: target = 'renal_failure' o target = 'IR'
df_merged['Grupo'] = df_merged[target].map({True: 'Insuficiencia Renal', False: 'Sin Insuficiencia Renal'})

# Crear subplots
n_cols = 5
n_rows = len(lab_columns) // n_cols + (len(lab_columns) % n_cols > 0)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

# Generar cada gráfico
for i, lab_test in enumerate(lab_columns):
    ax = axes[i]
    
    sns.violinplot(
        x='Grupo', y=lab_test, data=df_merged, ax=ax, inner="box",
        order=['Insuficiencia Renal', 'Sin Insuficiencia Renal'],
        palette=['#5481a6', '#3f3faf']
    )

    # Títulos y ejes
    nombre_legible = nombres_legibles[lab_test]
    unidad = unidades_lab[lab_test]
    ax.set_title(f'Distribución de {nombre_legible}', fontsize=12)
    ax.set_xlabel('')
    ax.set_ylabel(f'{nombre_legible} ({unidad})')

    # Líneas de rango normal
    rango_inf, rango_sup = rangos_lab[lab_test]
    ax.axhline(rango_inf, color='gray', linestyle='--', linewidth=1)
    ax.axhline(rango_sup, color='gray', linestyle='--', linewidth=1)
    ax.text(0.5, rango_inf, f'↓ {rango_inf}', ha='left', va='bottom', fontsize=8, color='black')
    ax.text(0.5, rango_sup, f'↑ {rango_sup}', ha='left', va='bottom', fontsize=8, color='black')

# Eliminar ejes vacíos si los hay
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Título general
fig.suptitle('Distribución de los principales electrolitos de pacientes DT1 con y sin IR', fontsize=16, y=1.02)

# Ajustes finales
plt.tight_layout()
plt.savefig('violin_plot_electrolitos_con_rangos_normales.png', bbox_inches='tight')
plt.show()


In [ ]:
import seaborn as sns 
import matplotlib.pyplot as plt

# Diccionario de nombres legibles
nombres_legibles = {
    'Hematocrit-Blood-Hematology': 'Hematocrito',
    'Hemoglobin-Blood-Hematology': 'Hemoglobina',
    'Magnesium-Blood-Chemistry': 'Magnesio'
}

# Unidades por prueba
unidades_lab = {
    'Hematocrit-Blood-Hematology': '%',
    'Hemoglobin-Blood-Hematology': 'g/dL',
    'Magnesium-Blood-Chemistry': 'mg/dL'
}

# Rangos normales por prueba
rangos_lab = {
    'Hematocrit-Blood-Hematology': (38.0, 50.0),
    'Hemoglobin-Blood-Hematology': (13.5, 17.5),
    'Magnesium-Blood-Chemistry': (1.5, 2.5)
}

# Columnas a graficar
lab_columns = list(nombres_legibles.keys())

# Asegúrate de tener esta columna bien definida
df_merged['Grupo'] = df_merged[target].map({True: 'Insuficiencia Renal', False: 'Sin Insuficiencia Renal'})

# Crear subplots
n_cols = 3
fig, axes = plt.subplots(1, n_cols, figsize=(n_cols * 6, 5))
axes = axes.flatten()

# Generar cada gráfico
for i, lab_test in enumerate(lab_columns):
    ax = axes[i]
    
    sns.violinplot(
        x='Grupo', y=lab_test, data=df_merged, ax=ax, inner="box",
        order=['Insuficiencia Renal', 'Sin Insuficiencia Renal'],
        palette=['#5481a6', '#3f3faf']
    )

    # Títulos y ejes
    nombre_legible = nombres_legibles[lab_test]
    unidad = unidades_lab[lab_test]
    ax.set_title(f'Distribución de {nombre_legible}', fontsize=12)
    ax.set_xlabel('')
    ax.set_ylabel(f'{nombre_legible} ({unidad})')

    # Líneas de rango normal
    rango_inf, rango_sup = rangos_lab[lab_test]
    ax.axhline(rango_inf, color='gray', linestyle='--', linewidth=1)
    ax.axhline(rango_sup, color='gray', linestyle='--', linewidth=1)

# Título general
fig.suptitle('Distribución de parámetros hematológicos seleccionados en pacientes DT1 con y sin IR', fontsize=16, y=1.05)

# Ajustes finales
plt.tight_layout()
plt.savefig('violin_plot_hematologicos_con_rangos.png', bbox_inches='tight')
plt.show()


In [ ]:
import seaborn as sns 
import matplotlib.pyplot as plt

# Diccionario de nombres legibles
nombres_legibles = {
    'Creatinine-Blood-Chemistry': 'Creatinina',
    'Urea Nitrogen-Blood-Chemistry': 'Nitrógeno ureico (BUN)'
}

# Unidades por prueba
unidades_lab = {
    'Creatinine-Blood-Chemistry': 'mg/dL',
    'Urea Nitrogen-Blood-Chemistry': 'mg/dL'
}

# Rangos normales por prueba
rangos_lab = {
    'Creatinine-Blood-Chemistry': (0.5, 1.2),
    'Urea Nitrogen-Blood-Chemistry': (7.0, 20.0)
}

lab_columns = list(nombres_legibles.keys())
df_merged['Grupo'] = df_merged[target].map({True: 'Insuficiencia Renal', False: 'Sin Insuficiencia Renal'})

# Crear subplots
n_cols = 2
fig, axes = plt.subplots(1, n_cols, figsize=(n_cols * 6, 5))
axes = axes.flatten()

for i, lab_test in enumerate(lab_columns):
    ax = axes[i]
    
    sns.violinplot(
        x='Grupo', y=lab_test, data=df_merged, ax=ax, inner="box",
        order=['Insuficiencia Renal', 'Sin Insuficiencia Renal'],
        palette=['#5481a6', '#3f3faf']
    )

    nombre_legible = nombres_legibles[lab_test]
    unidad = unidades_lab[lab_test]
    ax.set_title(f'Distribución de {nombre_legible}', fontsize=12)
    ax.set_xlabel('')
    ax.set_ylabel(f'{nombre_legible} ({unidad})')

    # Líneas de rango normal
    rango_inf, rango_sup = rangos_lab[lab_test]
    ax.axhline(rango_inf, color='black', linestyle='--', linewidth=1)
    ax.axhline(rango_sup, color='black', linestyle='--', linewidth=1)

    # Ajustar los límites del eje Y para asegurar que se ven los rangos
    min_val = df_merged[lab_test].min()
    max_val = df_merged[lab_test].max()
    margen = (max_val - min_val) * 0.05
    ax.set_ylim(min(min_val, rango_inf) - margen, max(max_val, rango_sup) + margen)

# Título general
fig.suptitle('Distribución de creatinina y BUN en pacientes DT1 con y sin IR', fontsize=16, y=1.05)

plt.tight_layout()
plt.savefig('violin_plot_creatinina_bun_con_rangos.png', bbox_inches='tight')
plt.show()


### Medicación

In [ ]:
med_columns=list(df_med.columns)
med_columns.remove('patient_id')


#### 1) Pacientes que toman algún tipo de medicación vs los que no toman ninguna


In [ ]:
# Convertir 'patient_id' de df_med a lista
med_pat_list = df_med['patient_id'].unique()

# Asegurarnos de que ambos estén en el mismo tipo de datos (como strings)
patient_ids_in_df = df_merged['patient_id'].unique().astype(str).tolist()
med_pat_list = med_pat_list.astype(str)

# Identificar los pacientes en df que están en la lista med_pat_list
pacientes_en_dataset = [patient for patient in med_pat_list if patient in patient_ids_in_df]

# Pacientes de la lista que no están en el dataset
pacientes_no_en_dataset = [patient for patient in med_pat_list if patient not in patient_ids_in_df]

# Mostrar el número de pacientes que no están en el dataset
print(f"Pacientes en la lista que no están en el dataset final: {len(pacientes_no_en_dataset)}")
print(f"Pacientes en la lista que están en el dataset final: {len(pacientes_en_dataset)}")

#### 2) El medicamento más común entre los pacientes que toman medicación:


In [ ]:
# Crear la nueva columna 'toma medicacion' en df_merged
df_merged['toma medicacion'] = df_merged['patient_id'].astype(str).isin(pacientes_en_dataset)
med_pats=df_merged[df_merged['toma medicacion']==True]
no_med_pats=df_merged[df_merged['toma medicacion']==False]

In [ ]:
med_pats.head()

In [ ]:
# Filtrar los pacientes que toman medicación (no nulos ni ceros)
df_medication_taken = med_pats[med_pats[med_columns].gt(0).any(axis=1)]

# Contar cuántos pacientes toman cada medicamento (sumamos 1 por cada paciente único)
medication_counts = (df_medication_taken[med_columns] > 0).sum(axis=0).sort_values(ascending=False)

# Obtener el medicamento más común
most_common_medication = medication_counts.idxmax()
print(f"El medicamento más común es: {most_common_medication}")
print(f"Con {medication_counts.max()} pacientes tomando este medicamento")

# Mostrar los 20 medicamentos más comunes
top_20_medications = medication_counts.head(20)
print("Los 20 medicamentos más comunes son:")
print(top_20_medications)



### 3- Análisis de relaciones

#### Medicación pacientes diabéticos con IR vs sin IR

Hipótesis: los pacientes DT1_IR toman mas medicacion para regualr el metabolismo, los fallos renales y las complicaciones cardiacas que los que no tiene IR.

In [ ]:
df_renal=df_merged[df_merged[target]==True]
df_no_renal=df_merged[df_merged[target]==False]

In [ ]:
# Contamos cuántos pacientes con insuficiencia renal toman y no toman medicación
renal_medication = df_renal[df_renal['toma medicacion'] == True].shape[0]
renal_no_medication = df_renal[df_renal['toma medicacion'] == False].shape[0]

# Contamos cuántos pacientes diabéticos tipo 1 (sin considerar insuficiencia renal) toman y no toman medicación
total_medication = df_merged[df_merged['toma medicacion'] == True].shape[0]
total_no_medication = df_merged[df_merged['toma medicacion'] == False].shape[0]

# Crear la tabla de comparación
medication_table = pd.DataFrame({
    'Medicación': [renal_medication, total_medication - renal_medication],
    'Sin Medicación': [renal_no_medication, total_no_medication - renal_no_medication]
}, index=['Pacientes DT1_IR', 'Pacientes DT1'])


# Ver también las primeras filas si prefieres verlo en el código
medication_table


In [ ]:
# Filtrar los pacientes que toman medicación en ambos grupos
df_renal_med = df_renal[df_renal['toma medicacion'] == True]
df_no_renal_med = df_no_renal[df_no_renal['toma medicacion'] == True]

# Contar cuántos pacientes toman cada medicamento en los dos grupos (con IR y sin IR)
medication_counts_renal = (df_renal_med[med_columns] > 0).sum(axis=0)
medication_counts_no_renal = (df_no_renal_med[med_columns] > 0).sum(axis=0)

# Crear una tabla de frecuencias con la cantidad de pacientes para cada medicamento
medication_comparison = pd.DataFrame({
    'Con IR': medication_counts_renal,
    'Sin IR': medication_counts_no_renal
}).fillna(0)

# Calcular las frecuencias normalizadas (para tener en cuenta el número diferente de pacientes en ambos grupos)
total_renal = len(df_renal_med)
total_no_renal = len(df_no_renal_med)

medication_comparison['Frecuencia con IR'] = 100 * medication_comparison['Con IR'] / total_renal
medication_comparison['Frecuencia sin IR'] = 100 * medication_comparison['Sin IR'] / total_no_renal

# Ordenar la tabla según la frecuencia en pacientes con insuficiencia renal
medication_comparison_sorted = medication_comparison.sort_values(by='Frecuencia con IR', ascending=False).head(15)

# Resaltar filas donde la frecuencia con IR es mayor que sin IR
styled_table = medication_comparison_sorted.style.apply(
    lambda x: ['background-color: #5481a6' if x['Frecuencia con IR'] > x['Frecuencia sin IR'] else '' for i in range(len(x))], axis=1
)

styled_table



In [ ]:
# Filtrar los pacientes que toman medicación en ambos grupos
df_renal_med = df_renal[df_renal['toma medicacion'] == True]
df_no_renal_med = df_no_renal[df_no_renal['toma medicacion'] == True]

# Contar cuántos pacientes toman cada medicamento en los dos grupos (con IR y sin IR)
medication_counts_renal = (df_renal_med[med_columns] > 0).sum(axis=0)
medication_counts_no_renal = (df_no_renal_med[med_columns] > 0).sum(axis=0)

# Crear una tabla de frecuencias con la cantidad de pacientes para cada medicamento
medication_comparison = pd.DataFrame({
    'Con IR': medication_counts_renal,
    'Sin IR': medication_counts_no_renal
}).fillna(0)

# Calcular las frecuencias normalizadas (para tener en cuenta el número diferente de pacientes en ambos grupos)
total_renal = len(df_renal_med)
total_no_renal = len(df_no_renal_med)

medication_comparison['Frecuencia con IR'] = 100 * medication_comparison['Con IR'] / total_renal
medication_comparison['Frecuencia sin IR'] = 100 * medication_comparison['Sin IR'] / total_no_renal

# Ordenar la tabla según la frecuencia en pacientes con insuficiencia renal
medication_comparison_sorted = medication_comparison.sort_values(by='Frecuencia con IR', ascending=False)

# Resaltar filas donde la frecuencia con IR es mayor que sin IR
styled_table = medication_comparison_sorted.style.apply(
    lambda x: ['background-color: #5481a6' if x['Frecuencia con IR'] > x['Frecuencia sin IR'] else '' for i in range(len(x))], axis=1
)

styled_table


In [ ]:
import matplotlib.pyplot as plt

# Crear un gráfico de barras para visualizar las frecuencias de medicamentos
plt.figure(figsize=(14, 8))

# Crear las barras para la frecuencia con IR y sin IR
medication_comparison_sorted[['Frecuencia con IR', 'Frecuencia sin IR']].plot(kind='bar', stacked=False, width=0.8, ax=plt.gca(), color=['#5481a6','#3f3faf'])

# Etiquetas y título
plt.title('Comparación de la Frecuencia de Medicamentos entre Pacientes con y sin Insuficiencia Renal', fontsize=16)
plt.xlabel('Medicamentos', fontsize=12)
plt.ylabel('Frecuencia (%)', fontsize=12)
plt.xticks(rotation=90, fontsize=10)
plt.legend(['Con Insuficiencia Renal', 'Sin Insuficiencia Renal'], loc='upper right')

# Ajustar el gráfico
plt.tight_layout()
plt.savefig('medicamentos_mayor_frecuencia_IR_.png')

# Mostrar el gráfico
plt.show()


In [ ]:
medication_comparison_sorted['Diferencia']=medication_comparison['Frecuencia con IR']-medication_comparison['Frecuencia sin IR']
medication_IR_mayor_noIR=medication_comparison_sorted[medication_comparison_sorted['Diferencia']>0]
medication_IR_mayor_noIR.sort_values(by='Frecuencia con IR', ascending=False)


In [ ]:
medication_IR_mayor_noIR.index

In [ ]:
import matplotlib.pyplot as plt

# Crear la figura y el eje
fig, ax = plt.subplots(figsize=(14, 8))

# Colores personalizados
colors = ['#5481a6', '#242465']

# Dibujar las barras
bars = medication_IR_mayor_noIR[['Frecuencia con IR', 'Frecuencia sin IR']].plot(
    kind='bar',
    stacked=False,
    width=0.8,
    ax=ax,
    color=colors
)

# Añadir etiquetas de datos con ajuste de posición para evitar solapamiento
for i, (idx, row) in enumerate(medication_IR_mayor_noIR.iterrows()):
    ax.text(i - 0.15, row['Frecuencia con IR'] + 1, f"{row['Frecuencia con IR']:.1f}", 
            color='black', ha='center', fontsize=8, rotation=90)
    ax.text(i + 0.15, row['Frecuencia sin IR'] + 1, f"{row['Frecuencia sin IR']:.1f}", 
            color='black', ha='center', fontsize=8, rotation=90)

# Etiquetas y título
ax.set_title('Medicación MÁS frecuente en pacientes con IR que sin IR', fontsize=16)
ax.set_xlabel('Medicamentos', fontsize=12)
ax.set_ylabel('Frecuencia (%)', fontsize=12)
ax.set_xticks(range(len(medication_IR_mayor_noIR)))
ax.set_xticklabels(medication_IR_mayor_noIR.index, rotation=90, fontsize=9)
ax.legend(['Con Insuficiencia Renal', 'Sin Insuficiencia Renal'], loc='upper right')

# Ajustar el diseño
plt.tight_layout()
plt.savefig('medicamentos_mayor_frecuencia_IR_.png')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Grupos clave
grupo_metabolismo = [
    'Endocrine & Metabolic', 'Vitamin', 'Supplement', 'Iron Supplements', 'Antigout'
]
grupo_renal = [
    'Renal', 'Electrolyte', 'Fluid & Electrolyte', 'Diuretic'
]
grupo_cardiovascular = [
    'Antihypertensive', 'Lipid-lowering', 'Antiplatelet', 'Anticoagulant',
    'Thrombolytic', 'Vasodilator', 'Cardiovascular', 'Vasopressor'
]

# Lista total de medicamentos a mostrar
meds_mostrar = (
    [med for med in grupo_metabolismo if med in medication_IR_mayor_noIR.index] +
    [med for med in grupo_renal if med in medication_IR_mayor_noIR.index] +
    [med for med in grupo_cardiovascular if med in medication_IR_mayor_noIR.index]
)

# Opcional: añadir unos pocos "otros" representativos
otros = ['Antibiotic', 'Respiratory', 'Laxative']
otros = [med for med in otros if med in medication_IR_mayor_noIR.index]
meds_mostrar += otros

# Filtrar y ordenar
medication_filtrado = medication_IR_mayor_noIR.loc[meds_mostrar]

# Grupo dict para etiquetas
grupo_dict = {med: 'Metabolismo' for med in grupo_metabolismo}
grupo_dict.update({med: 'Tratamiento renal' for med in grupo_renal})
grupo_dict.update({med: 'Riesgo cardiovascular' for med in grupo_cardiovascular})
grupo_dict.update({med: 'Otros' for med in otros})

# Graficar
fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#004c6d', '#9cb2d6']
bars = medication_filtrado[['Frecuencia con IR', 'Frecuencia sin IR']].plot(
    kind='bar', stacked=False, width=0.8, ax=ax, color=colors
)

# Etiquetas de porcentaje
for i, (idx, row) in enumerate(medication_filtrado.iterrows()):
    ax.text(i - 0.15, row['Frecuencia con IR'] + 1, f"{row['Frecuencia con IR']:.1f}",
            color='black', ha='center', fontsize=8, rotation=90)
    ax.text(i + 0.15, row['Frecuencia sin IR'] + 1, f"{row['Frecuencia sin IR']:.1f}",
            color='black', ha='center', fontsize=8, rotation=90)

# Agrupar con etiquetas
xticks = list(medication_filtrado.index)
group_positions = {'Metabolismo': [], 'Tratamiento renal': [], 'Riesgo cardiovascular': [], 'Otros': []}

for i, med in enumerate(xticks):
    grupo = grupo_dict.get(med)
    if grupo:
        group_positions[grupo].append(i)

# Etiquetas de grupo debajo del eje X
ax.set_xticks(range(len(xticks)))
ax.set_xticklabels(xticks, rotation=90, fontsize=9)

ymin, ymax = ax.get_ylim()
y_label = -6

for grupo, pos in group_positions.items():
    if pos:
        inicio = pos[0] - 0.4
        fin = pos[-1] + 0.4
        medio = (inicio + fin) / 2
        ax.hlines(y=y_label + 1, xmin=inicio, xmax=fin, color='black', linewidth=1.5)
        ax.vlines(x=inicio, ymin=y_label + 0.5, ymax=y_label + 1, color='black', linewidth=1.5)
        ax.vlines(x=fin, ymin=y_label + 0.5, ymax=y_label + 1, color='black', linewidth=1.5)
        ax.text(medio, y_label, grupo, ha='center', va='top', fontsize=10, fontweight='bold')

# Ajustes finales
ax.set_ylim(ymin - 12, ymax + 5)
ax.set_title('Medicación MÁS frecuente en pacientes con IR que sin IR', fontsize=16)
ax.set_xlabel('Medicamentos', fontsize=12)
ax.set_ylabel('Frecuencia (%)', fontsize=12)
ax.legend(['Con Insuficiencia Renal', 'Sin Insuficiencia Renal'], loc='upper right')

plt.tight_layout()
plt.savefig('medicacion_filtrada_grupos.png')
plt.show()



In [ ]:
medication_comparison_sorted['Diferencia']=medication_comparison['Frecuencia con IR']-medication_comparison['Frecuencia sin IR']
medication_IR_menor_noIR=medication_comparison_sorted[medication_comparison_sorted['Diferencia']<0]
medication_IR_menor_noIR.sort_values(by='Frecuencia con IR', ascending=False)

In [ ]:
medication_IR_menor_noIR.index

In [ ]:
import matplotlib.pyplot as plt

# Crear la figura y el eje
fig, ax = plt.subplots(figsize=(14, 8))

# Colores personalizados
colors = ['#5481a6', '#242465']

# Dibujar las barras
bars = medication_IR_menor_noIR[['Frecuencia con IR', 'Frecuencia sin IR']].plot(
    kind='bar',
    stacked=False,
    width=0.8,
    ax=ax,
    color=colors
)

# Añadir etiquetas de datos
for i, (idx, row) in enumerate(medication_IR_menor_noIR.iterrows()):
    ax.text(i - 0.2, row['Frecuencia con IR'] + 1, f"{row['Frecuencia con IR']:.1f}", color='black', ha='center', fontsize=9)
    ax.text(i + 0.25, row['Frecuencia sin IR'] + 1, f"{row['Frecuencia sin IR']:.1f}", color='black', ha='center', fontsize=9)

# Etiquetas y título
ax.set_title('Medicación MENOS frecuente en pacientes CON IR que SIN IR', fontsize=16)
ax.set_xlabel('Medicamentos', fontsize=12)
ax.set_ylabel('Frecuencia (%)', fontsize=12)
ax.set_xticks(range(len(medication_IR_menor_noIR)))
ax.set_xticklabels(medication_IR_menor_noIR.index, rotation=90, fontsize=10)
ax.legend(['Con Insuficiencia Renal', 'Sin Insuficiencia Renal'], loc='upper right')

# Ajustar el diseño
plt.tight_layout()
plt.savefig('medicamentos_menor_frecuencia_IR_.png')


# Mostrar el gráfico
plt.show()



In [ ]:
import matplotlib.pyplot as plt

# Grupos definidos
grupo_dolor = ['Analgesic', 'Topical Analgesic', 'Opioid Agonist-Antagonist', 'Antiseptic']
grupo_mental = ['Antidepressant', 'Anxiolytic', 'Antipsychotic', 'Antiparkinsonian']
grupo_respiratorio = ['Antiasthmatic', 'Antihistamine', 'Vaccine', 'Smoking cessation']
grupo_otros = ['Hepatobiliary', 'Urological', 'Obstetrics', 'Antiflatulent', 'Radiology', 'Antagonist']

# Crear diccionario de grupos
grupo_dict = {}
for med in grupo_dolor:
    grupo_dict[med] = 'Dolor y anestesia'
for med in grupo_mental:
    grupo_dict[med] = 'Salud mental'
for med in grupo_respiratorio:
    grupo_dict[med] = 'Respiratorio e inmunológico'
for med in grupo_otros:
    grupo_dict[med] = 'Otros'

# Reordenar el DataFrame según el orden de grupos
orden = grupo_dolor + grupo_mental + grupo_respiratorio + grupo_otros
orden = [med for med in orden if med in medication_IR_menor_noIR.index]
medication_ordenado = medication_IR_menor_noIR.loc[orden]

# Crear figura
fig, ax = plt.subplots(figsize=(16, 8))

# Dibujar barras
colors = ['#5481a6', '#242465']
bars = medication_ordenado[['Frecuencia con IR', 'Frecuencia sin IR']].plot(
    kind='bar',
    stacked=False,
    width=0.8,
    ax=ax,
    color=colors
)

# Etiquetas de porcentaje
for i, (idx, row) in enumerate(medication_ordenado.iterrows()):
    ax.text(i - 0.15, row['Frecuencia con IR'] + 0.5, f"{row['Frecuencia con IR']:.1f}", 
            color='black', ha='center', fontsize=8, rotation=90)
    ax.text(i + 0.15, row['Frecuencia sin IR'] + 0.5, f"{row['Frecuencia sin IR']:.1f}", 
            color='black', ha='center', fontsize=8, rotation=90)

# Agrupación visual debajo del eje X
xticks = list(medication_ordenado.index)
group_positions = {'Dolor y anestesia': [], 'Salud mental': [], 'Respiratorio e inmunológico': [], 'Otros': []}

for i, med in enumerate(xticks):
    grupo = grupo_dict.get(med)
    if grupo:
        group_positions[grupo].append(i)

ax.set_xticks(range(len(xticks)))
ax.set_xticklabels(xticks, rotation=90, fontsize=9)

ymin, ymax = ax.get_ylim()
y_label = -5

for grupo, pos in group_positions.items():
    if pos:
        inicio = pos[0] - 0.4
        fin = pos[-1] + 0.4
        medio = (inicio + fin) / 2
        ax.hlines(y=y_label + 1, xmin=inicio, xmax=fin, color='black', linewidth=1.5)
        ax.vlines(x=inicio, ymin=y_label + 0.5, ymax=y_label + 1, color='black', linewidth=1.5)
        ax.vlines(x=fin, ymin=y_label + 0.5, ymax=y_label + 1, color='black', linewidth=1.5)
        ax.text(medio, y_label, grupo, ha='center', va='top', fontsize=10, fontweight='bold')

# Ajustar los límites verticales
ax.set_ylim(ymin - 10, ymax + 5)

# Título y leyenda
ax.set_title('Medicación MENOS frecuente en pacientes CON IR que SIN IR', fontsize=16)
ax.set_xlabel('Medicamentos', fontsize=12)
ax.set_ylabel('Frecuencia (%)', fontsize=12)
ax.legend(['Con Insuficiencia Renal', 'Sin Insuficiencia Renal'], loc='upper right')

plt.tight_layout()
plt.savefig('medicacion_menos_frecuente_grupos.png')
plt.show()


### 4- Feature Engineering

### 41- Eliminamos las variables numéricas con varianza 0

In [ ]:
# Eliminar variables constantes
df_merged = df_merged.drop(columns=cte_vars)


In [ ]:
df_merged.head()

In [ ]:
df_merged.shape[0]

### 2- Creamos nueva variable: murio (True/False)

In [ ]:
# Crear variable booleana 'murio' según dod y deathtime
df_merged['murio'] = df_merged[['dod', 'deathtime']].notnull().any(axis=1)


In [ ]:
print('Han muerto', df_merged[df_merged['murio']==True].shape[0], 'pacientes')

#### Valores nulos

##### De los pacientes que toman medicación (995/2220)

In [ ]:
# Filtrar pacientes que toman medicación
med_pats = df_merged[df_merged['toma medicacion'] == True]

# Ver nulos y su porcentaje
nulos_med = med_pats.isnull().sum()
nulos_pct = (nulos_med / len(med_pats)) * 100
nulos_filtrados = nulos_pct[nulos_pct > 0].sort_values(ascending=False)

print(nulos_filtrados)


Hay muchos pacientes del grupo "toma_medicacion"=TRUE CON VALORES NULOS asi que vamos a seguir este workflow:
    
- Caso 1: >95% de nulos:
        - Binarizacion (toma_xxxx):Iron Supplements, Cardiovascular
        - Eliminar: Hepatobiliary, Anticholinergic, Antispasmodic, Antiparkinsonian, Obstetrics, Antigout, Stimulant, Antidote, Expectorant, Muscle relaxant
        
- Caso 2: 70-95% de nulos:
        - Binarizar:Renal, Corticosteroid, Diuretic, Vasodilator, Vasopressor, Anesthetic, Supplement, Urological, Immunosupressant
        - Eliminar: Antacid, Ophthalmic, Gastrointestinal, Antiviral, Antifungal, Oral Care, H2 Antihistamine, Local Anesthetic, Vitamin, Anticonvulsant, Anxiolytic

- Caso 3: 30-69% de nulos
> Imputar: Antiplatelet, Fluid & Electrolyte, Lipid-lowering, Antibiotic
> Binarizar:Antidepressant, Proton Pump Inhibitor
> Eliminar: Antiemetic

- Caso 4: <30% nulos:
        - Imputar: Antihypertensive, Analgesic, Anticoagulant, Endocrine & Metabolic, Electrolyte


In [ ]:
##ANTIGUO

# df_meds_procesado = med_pats.copy()


# # Listas de variables a eliminar 
# variables_eliminar = [
#     'Hepatobiliary','Anticholinergic', 'Antispasmodic', 'Antiparkinsonian', 'Obstetrics', 'Antigout',
#     'Stimulant', 'Antidote', 'Expectorant', 'Muscle relaxant', 'Antacid', 'Ophthalmic',
#     'Gastrointestinal', 'Antiviral', 'Antifungal', 'Oral Care',
#     'Local Anesthetic', 'Vitamin', 'Anticonvulsant', 'Anxiolytic', 'Antiemetic'
# ]

# # Variables a binarizar (crear columna "toma_X" = 1 si no nulo y > 0, 0 en caso contrario)
# variables_binarizar = [
#     'Iron Supplements', 'Cardiovascular', 'Renal', 'Corticosteroid', 'Diuretic',
#     'Vasodilator', 'Vasopressor', 'Anesthetic', 'Supplement',
#     'Antipsychotic', 'Proton Pump Inhibitor','Urological', 'Immunosuppressant', 'Sedative', 'Antihistamine', 'H2 Antihistamine'
# ]

# # Eliminar columnas irrelevantes
# df_meds_procesado.drop(columns=variables_eliminar, inplace=True, errors='ignore')

# # Crear columnas binarizadas
# for var in variables_binarizar:
#     nueva_col = f'toma_{var.lower().replace(" ", "_")}'
#     df_meds_procesado[nueva_col] = df_meds_procesado[var].notnull() & (df_meds_procesado[var] > 0)
#     df_meds_procesado[nueva_col] = df_meds_procesado[nueva_col].astype(int)
    

In [ ]:
# df_meds_procesado.head()

In [ ]:
df_meds_procesado = med_pats.copy()
df_merged_copy=df_merged.copy()


# Listas de variables a eliminar 
variables_eliminar = [
    'Hepatobiliary','Anticholinergic', 'Antispasmodic', 'Antiparkinsonian', 'Obstetrics', 'Antigout',
    'Stimulant', 'Antidote', 'Expectorant', 'Muscle relaxant', 'Antacid', 'Ophthalmic',
    'Gastrointestinal', 'Antiviral', 'Antifungal', 'Oral Care',
    'Local Anesthetic', 'Vitamin', 'Anticonvulsant', 'Anxiolytic', 'Antiemetic'
]

# Variables a binarizar (crear columna "toma_X" = 1 si no nulo y > 0, 0 en caso contrario)
variables_binarizar = [
    'Iron Supplements', 'Cardiovascular', 'Renal', 'Corticosteroid', 'Diuretic',
    'Vasodilator', 'Vasopressor', 'Anesthetic', 'Supplement',
    'Antipsychotic', 'Proton Pump Inhibitor','Urological', 'Immunosuppressant', 'Sedative', 'Antihistamine', 'H2 Antihistamine'
]

# Eliminar columnas irrelevantes
df_merged_copy.drop(columns=variables_eliminar, inplace=True, errors='ignore')

# Crear columnas binarizadas
for var in variables_binarizar:
    nueva_col = f'toma_{var.lower().replace(" ", "_")}'
    df_merged_copy[nueva_col] = df_merged_copy[var].notnull() & (df_merged_copy[var] > 0)
    df_merged_copy[nueva_col] = df_merged_copy[nueva_col].astype(int)

In [ ]:
df_merged_copy.head()

In [ ]:
# Mostrar lista de columnas esperadas para validación
max_len = max(len(variables_eliminar), len(variables_binarizar))

# Rellenar con None para igualar longitud
variables_eliminar += [None] * (max_len - len(variables_eliminar))
variables_binarizar += [None] * (max_len - len(variables_binarizar))

# Crear DataFrame organizado
df_resultado = pd.DataFrame({
    "Variables a eliminar": variables_eliminar,
    "Variables a binarizar": variables_binarizar
})
df_resultado

In [ ]:
# #ANTIGUO

# #Caso 3: Imputar columnas 30–69 % nulos que son menos críticos con la mediana
# from sklearn.impute import SimpleImputer

# # Crear copia de trabajo
# df_imputado = df_meds_procesado.copy()

# # Variables del caso 3 (30–69% nulos): imputar con mediana
# variables_caso3_mediana = [
#     'Antiplatelet', 'Fluid & Electrolyte', 'Lipid-lowering', 'Antibiotic', 'Antidepressant'
# ]

# # Filtrar las que existen y son numéricas
# variables_caso3_validas = [
#     col for col in variables_caso3_mediana 
#     if col in df_imputado.columns and pd.api.types.is_numeric_dtype(df_imputado[col])
# ]

# # Aplicar SimpleImputer con mediana
# if variables_caso3_validas:
#     imputer = SimpleImputer(strategy='median')
#     df_imputado[variables_caso3_validas] = imputer.fit_transform(df_imputado[variables_caso3_validas])



In [ ]:
# df_imputado.head()

In [ ]:
# # Excluir variables binarizadas (todas las que empiezan por "toma_")
# columnas_no_binarizadas = [col for col in df_imputado.columns if not variables_binarizar ]

# # Calcular porcentaje de nulos solo en variables no binarizadas
# null_percentages_filtrado = df_imputado[columnas_no_binarizadas].isnull().mean() * 100

# # Filtrar columnas con nulos
# null_status_filtrado = null_percentages_filtrado[null_percentages_filtrado > 0].sort_values(ascending=False).to_frame(name='Porcentaje_nulos')
# null_status_filtrado['Tipo_de_dato'] = df_imputado[null_status_filtrado.index].dtypes.values

# null_status_filtrado

In [ ]:
# Lista de columnas a revisar
columns = [
    'Analgesic', 'Antibiotic', 'Anticoagulant', 'Antidepressant',
    'Antihypertensive', 'Antiplatelet', 'BUN/Creatinina', 'eGFR',
    'Electrolyte', 'Fluid & Electrolyte', 'Lipid-lowering'
]

# Contar valores nulos en las columnas especificadas
for col in columns:
    if col in df_imputado_general.columns:
        null_count = df_imputado_general[col].isnull().sum()
        print(f"Columna {col}: {null_count} valores nulos")


In [ ]:
from sklearn.impute import SimpleImputer
import pandas as pd

# Crear copia de trabajo
df_imputado_general = df_merged_copy.copy()

# Variables del caso 3 (30–69% nulos): imputar con mediana
variables_caso3_mediana = [
    'Antiplatelet', 'Fluid & Electrolyte', 'Lipid-lowering', 'Antibiotic', 'Antidepressant'
]

# Filtrar las que existen y son numéricas
variables_caso3_validas = [
    col for col in variables_caso3_mediana 
    if col in df_imputado_general.columns and pd.api.types.is_numeric_dtype(df_imputado_general[col])
]

# Imputar con la mediana para las variables válidas
if variables_caso3_validas:
    imputer = SimpleImputer(strategy='median')
    df_imputado_general[variables_caso3_validas] = imputer.fit_transform(df_imputado_general[variables_caso3_validas])

# Imputar con 0 en las filas donde 'toma medicacion' es False
df_imputado_general.loc[df_imputado_general['toma medicacion'] == False, variables_caso3_validas] = pd.NA


In [ ]:
# Verificar si las filas con 'toma medicacion' == False han sido imputadas con 0
print(df_imputado_general.loc[df_imputado_general['toma medicacion'] == False, variables_caso3_validas].head())


In [ ]:
# Verificar las estadísticas de las columnas antes y después de la imputación
print("Estadísticas antes de imputar:")
print(df_merged_copy[variables_caso3_validas].isnull().sum())  # Antes de la imputación

print("\nEstadísticas después de imputar:")
print(df_imputado_general[variables_caso3_validas].isnull().sum())  # Después de la imputación


In [ ]:
# #Caso 3: Imputar columnas 30–69 % nulos que son menos críticos con la mediana
# from sklearn.impute import SimpleImputer

# # Crear copia de trabajo
# df_imputado_general = df_merged_copy.copy()

# # Variables del caso 3 (30–69% nulos): imputar con mediana
# variables_caso3_mediana = [
#     'Antiplatelet', 'Fluid & Electrolyte', 'Lipid-lowering', 'Antibiotic', 'Antidepressant'
# ]

# # Filtrar las que existen y son numéricas
# variables_caso3_validas = [
#     col for col in variables_caso3_mediana 
#     if col in df_imputado_general.columns and pd.api.types.is_numeric_dtype(df_imputado_general[col])
# ]

# # Aplicar SimpleImputer con mediana
# if variables_caso3_validas:
#     imputer = SimpleImputer(strategy='median')
#     df_imputado_general[variables_caso3_validas] = imputer.fit_transform(df_imputado_general[variables_caso3_validas])

In [ ]:
# Excluir variables binarizadas (todas las que empiezan por "toma_")
columnas_no_binarizadas = [col for col in  df_imputado_general.columns if not variables_binarizar ]

# Calcular porcentaje de nulos solo en variables no binarizadas
null_percentages_filtrado =  df_imputado_general[columnas_no_binarizadas].isnull().mean() * 100

# Filtrar columnas con nulos
null_status_filtrado = null_percentages_filtrado[null_percentages_filtrado > 0].sort_values(ascending=False).to_frame(name='Porcentaje_nulos')
null_status_filtrado['Tipo_de_dato'] =  df_imputado_general[null_status_filtrado.index].dtypes.values

null_status_filtrado

In [ ]:
# ANTIGUO

# # Caso 4: Imputar columnas con <30% de nulos usando KNNImputer

# from sklearn.impute import KNNImputer
# # Lista de columnas especificadas por Ana para imputar con KNN
# columnas_4= [
#     'Antihypertensive', 'Analgesic', 'Anticoagulant', 'Endocrine & Metabolic', 'Electrolyte'
# ]

# # Verificar que existen en el DataFrame y son numéricas
# columnas_4_existentes= [
#     col for col in columnas_4
#     if col in df_imputado.columns and pd.api.types.is_numeric_dtype(df_imputado[col])
# ]

# # Aplicar KNNImputer si hay columnas válidas
# if columnas_4_existentes:
#     knn_imputer = KNNImputer(n_neighbors=5)
#     df_imputado[columnas_4_existentes] = knn_imputer.fit_transform(df_imputado[columnas_4_existentes])


In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Seleccionar las columnas que se imputaron en el caso 4
# columnas_imputadas = [
#     'Antihypertensive', 'Analgesic', 'Anticoagulant', 'Endocrine & Metabolic', 'Electrolyte'
# ]

# # Verificar que existen en el DataFrame
# columnas_existentes = [col for col in columnas_imputadas if col in df_imputado.columns]

# # Graficar la distribución de cada variable
# for col in columnas_existentes:
#     plt.figure(figsize=(6, 3))
#     sns.histplot(df_imputado[col], kde=True, bins=30)
#     plt.title(f'Distribución de: {col}')
#     plt.xlabel('Valor')
#     plt.ylabel('Frecuencia')
#     plt.tight_layout()
#     plt.show()


In [ ]:
# # Caso 4: Imputar columnas con <30% de nulos usando KNNImputer

# from sklearn.impute import KNNImputer
# # Lista de columnas especificadas por Ana para imputar con KNN
# columnas_4= [
#     'Antihypertensive', 'Analgesic', 'Anticoagulant', 'Endocrine & Metabolic', 'Electrolyte'
# ]

# # Verificar que existen en el DataFrame y son numéricas
# columnas_4_existentes= [
#     col for col in columnas_4
#     if col in df_imputado_general.columns and pd.api.types.is_numeric_dtype(df_imputado_general[col])
# ]

# # Aplicar KNNImputer si hay columnas válidas
# if columnas_4_existentes:
#     knn_imputer = KNNImputer(n_neighbors=5)
#     df_imputado_general[columnas_4_existentes] = knn_imputer.fit_transform(df_imputado_general[columnas_4_existentes])


In [ ]:
from sklearn.impute import KNNImputer
import pandas as pd

# Lista de columnas especificadas por Ana para imputar con KNN
columnas_4 = [
    'Antihypertensive', 'Analgesic', 'Anticoagulant', 'Endocrine & Metabolic', 'Electrolyte'
]

# Verificar que existen en el DataFrame y son numéricas
columnas_4_existentes = [
    col for col in columnas_4
    if col in df_imputado_general.columns and pd.api.types.is_numeric_dtype(df_imputado_general[col])
]

# Imputar con KNNImputer si hay columnas válidas
if columnas_4_existentes:
    knn_imputer = KNNImputer(n_neighbors=5)
    df_imputado_general[columnas_4_existentes] = knn_imputer.fit_transform(df_imputado_general[columnas_4_existentes])

# Imputar con 0 en las filas donde 'toma medicacion' == False
df_imputado_general.loc[df_imputado_general['toma medicacion'] == False, columnas_4_existentes] = pd.NA

# Verificar que la imputación con 0 se ha realizado correctamente
print(df_imputado_general.loc[df_imputado_general['toma medicacion'] == False, columnas_4_existentes].head())

# Verificar las estadísticas de las columnas antes y después de la imputación
print("Estadísticas antes de imputar:")
print(df_merged_copy[columnas_4_existentes].isnull().sum())  # Antes de la imputación

print("\nEstadísticas después de imputar:")
print(df_imputado_general[columnas_4_existentes].isnull().sum())  # Después de la imputación


In [ ]:
# # Lista de variables que NO queremos incluir en la comprobación de nulos
# variables_binarizar_excluidas = [
#     'Iron Supplements', 'Cardiovascular', 'Renal', 'Corticosteroid', 'Diuretic',
#     'Vasodilator', 'Vasopressor', 'Anesthetic', 'Supplement',
#     'Antipsychotic', 'Proton Pump Inhibitor', 'Urological', 'Immunosuppressant',
#     'Sedative', 'Antihistamine', 'H2 Antihistamine'
# ]

# # Filtrar columnas que no están en esa lista
# columnas_a_revisar = [col for col in df_imputado.columns if col not in variables_binarizar_excluidas]

# # Comprobar nulos solo en esas columnas
# nulos_filtrados = df_imputado[columnas_a_revisar].isnull().sum()
# nulos_filtrados = nulos_filtrados[nulos_filtrados > 0].sort_values(ascending=False).to_frame(name='Nulos restantes')
# nulos_filtrados['% Nulos'] = (nulos_filtrados['Nulos restantes'] / len(df_imputado)) * 100
# nulos_filtrados['Tipo de dato'] = df_imputado[nulos_filtrados.index].dtypes.values
# nulos_filtrados

In [ ]:
# Lista de variables que NO queremos incluir en la comprobación de nulos
variables_binarizar_excluidas = [
    'Iron Supplements', 'Cardiovascular', 'Renal', 'Corticosteroid', 'Diuretic',
    'Vasodilator', 'Vasopressor', 'Anesthetic', 'Supplement',
    'Antipsychotic', 'Proton Pump Inhibitor', 'Urological', 'Immunosuppressant',
    'Sedative', 'Antihistamine', 'H2 Antihistamine'
]

# Filtrar columnas que no están en esa lista
columnas_a_revisar = [col for col in df_imputado_general.columns if col not in variables_binarizar_excluidas]

# Comprobar nulos solo en esas columnas
nulos_filtrados = df_imputado_general[columnas_a_revisar].isnull().sum()
nulos_filtrados = nulos_filtrados[nulos_filtrados > 0].sort_values(ascending=False).to_frame(name='Nulos restantes')
nulos_filtrados['% Nulos'] = (nulos_filtrados['Nulos restantes'] / len(df_imputado_general)) * 100
nulos_filtrados['Tipo de dato'] = df_imputado_general[nulos_filtrados.index].dtypes.values
nulos_filtrados

In [ ]:
# Lista de variables de medicación 
variables_medicacion = [
    "Analgesic", "Anesthetic", "Antacid", "Antagonist", "Antiasthmatic", "Antibiotic",
    "Anticholinergic", "Anticoagulant", "Anticonvulsant", "Antidepressant", "Antidiarrheal",
    "Antidote", "Antiemetic", "Antiflatulent", "Antifungal", "Antigout", "Antihistamine",
    "Antihypertensive", "Antiparkinsonian", "Antiplatelet", "Antipsychotic", "Antiseptic",
    "Antispasmodic", "Antitussive", "Antiviral", "Anxiolytic", "Cardiovascular", "Corticosteroid",
    "Diuretic", "Electrolyte", "Endocrine & Metabolic", "Expectorant", "Fluid & Electrolyte",
    "Gastrointestinal", "H2 Antihistamine", "Hepatobiliary", "Immunosuppressant",
    "Iron Supplements", "Laxative", "Lipid-lowering", "Local Anesthetic", "Muscle Relaxant",
    "Muscle relaxant", "Obstetrics", "Ophthalmic", "Opioid Agonist-Antagonist", "Oral Care",
    "Proton Pump Inhibitor", "Radiology", "Renal", "Respiratory", "Sedative",
    "Sedatives & Analgesics", "Smoking cessation", "Stimulant", "Supplement", "Thrombolytic",
    "Topical", "Topical Analgesic", "Topical Antibiotic", "Urological", "Vaccine",
    "Vasodilator", "Vasopressor", "Vitamin"
]

# Filtrar las columnas de medicación que existen en el DataFrame y son numéricas
columnas_medicacion_validas = [
    col for col in variables_medicacion
    if col in df_imputado.columns and pd.api.types.is_numeric_dtype(df_imputado[col])
]

# Crear subplots con estas columnas
import matplotlib.pyplot as plt
import seaborn as sns
import math

num_vars = len(columnas_medicacion_validas)
cols = 3
rows = math.ceil(num_vars / cols)

fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = axes.flatten()

for i, col in enumerate(columnas_medicacion_validas):
    sns.histplot(df_imputado[col], kde=True, bins=30, ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel("Valor")
    axes[i].set_ylabel("Frecuencia")

# Eliminar subplots vacíos
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()



## Hasta aqui la ultima copia con todos los cambios en medicación se llama df_imputado

### Crear variables egfr y bun/creatinina

Interpretación de resultados:
    
eGFR >90 → normal (si no hay otra enfermedad)

eGFR 60-89 → función renal levemente disminuida

eGFR 30-59 → insuficiencia renal moderada

eGFR 15-29 → insuficiencia renal severa

eGFR <15 → insuficiencia renal terminal

Interpretación de resultados:
    
- Si BUN/creatinina > 20, sugiere hipoperfusión renal (causas prerrenales, como deshidratación).

- Si BUN/creatinina ≈ 10-20, sugiere problema intrínseco renal (daño real en el riñón).



In [ ]:
# Crear copia de trabajo
df_variables_nuevas = df_imputado_general.copy()  #Cambiar al df completo que tb tenga en cuenta a los pacientes sin medicacion.

# Verificar nombres de columnas necesarias para el cálculo
col_creatinina = 'Creatinine-Blood-Chemistry'
col_bun = 'Urea Nitrogen-Blood-Chemistry'
col_edad = 'anchor_age'
col_sexo = 'gender'

# Crear la función de eGFR vectorizada
def calcular_egfr_vectorizado(creatinina, edad, sexo):
    sexo = sexo.astype(str) 
    k = sexo.map({'F': 0.7, 'M': 0.9})
    a = sexo.map({'F': -0.329, 'M': -0.411})
    factor_sexo = sexo.map({'F': 1.018, 'M': 1.0})
    
    cr_k = creatinina / k
    min_cr_k = cr_k.clip(upper=1)
    max_cr_k = cr_k.clip(lower=1)
    
    egfr = round(141 * (min_cr_k ** a) * (max_cr_k ** -1.209) * (0.993 ** edad) * factor_sexo, 2)
    return egfr

# Crear la variable eGFR
df_variables_nuevas['eGFR'] = calcular_egfr_vectorizado(
    df_variables_nuevas[col_creatinina],
    df_variables_nuevas[col_edad],
    df_variables_nuevas[col_sexo]
)

# Crear el ratio BUN/creatinina
df_variables_nuevas['BUN/Creatinina'] = round(df_variables_nuevas[col_bun] / df_variables_nuevas[col_creatinina], 2)

# Reemplazar divisiones inválidas con NaN (creatinina = 0)
df_variables_nuevas['BUN/Creatinina'].replace([float('inf'), -float('inf')], pd.NA, inplace=True)

# Asignar 0 a las nuevas columnas cuando 'toma medicacion' == False
df_variables_nuevas.loc[df_variables_nuevas['toma medicacion'] == False, ['eGFR', 'BUN/Creatinina']] = 0

# Verificar los resultados
#print(df_variables_nuevas.loc[df_variables_nuevas['toma medicacion'] == False, ['eGFR', 'BUN/Creatinina']].head())


In [ ]:
df_variables_nuevas.head()

In [ ]:
df_variables_nuevas[df_variables_nuevas['toma medicacion'] == False].head()

#### Visualicemos cuántos pacientes tienen estos valores alterados

In [ ]:
# Crear copia para trabajar con los resultados interpretados
df_resultados = df_variables_nuevas.copy()

# Recalcular eGFR y BUN/Creatinina (asumiendo que están presentes)
# Si no, estas líneas se deberían ejecutar previamente

# eGFR: crear columna de interpretación categórica
def interpretar_egfr(valor):
    if valor >= 90:
        return "Normal"
    elif 60 <= valor < 90:
        return "Función renal levemente disminuida"
    elif 30 <= valor < 60:
        return "Insuficiencia renal moderada"
    elif 15 <= valor < 30:
        return "Insuficiencia renal severa"
    else:
        return "Insuficiencia renal terminal"

df_resultados['eGFR_categoria'] = df_resultados['eGFR'].apply(interpretar_egfr)

# BUN/Creatinina: crear columna de interpretación
def interpretar_bun_creatinina(ratio):
    if pd.isna(ratio):
        return "Desconocido"
    elif ratio > 20:
        return "Niveles altos" # Causas: diabetes, deshidratación, insuficiencia cardíaca, dieta alta en proteinas o por ciertos medicamentos
    elif 10 <= ratio <= 20:
        return "Normal"
    elif ratio<10:
        return  "Niveles bajos" # Causas: Malnutrición, enfermedad hepática, dieta con bajo contenido de proteínas, a una lesión muscular grave (rabdomiólisis), al embarazo, a cirrosis o al síndrome de secreción inapropiada de hormona antidiurética (SIADH)
    else:
        return "Considerar otras causas"

df_resultados['BUN/Cr_interpretacion'] = df_resultados['BUN/Creatinina'].apply(interpretar_bun_creatinina)

# Generar conteos
conteo_egfr = df_resultados['eGFR_categoria'].value_counts().rename_axis('Categoría eGFR').to_frame('N pacientes')
conteo_egfr['%'] = round(100 * conteo_egfr['N pacientes'] / len(df_resultados), 2)

conteo_bun = df_resultados['BUN/Cr_interpretacion'].value_counts().rename_axis('Interpretación BUN/Cr').to_frame('N pacientes')
conteo_bun['%'] = round(100 * conteo_bun['N pacientes'] / len(df_resultados), 2)


In [ ]:
conteo_egfr

In [ ]:
conteo_bun

In [ ]:
df_variables_nuevas.columns

In [ ]:
df_variables_nuevas=df_variables_nuevas.set_index('patient_id')

In [ ]:
df_variables_nuevas.index = df_variables_nuevas.index.map(str)

In [ ]:
#Creamos nuevas variables para las variables que habiamos binarizado al principio:
medicamentos_consulta = [
    'Antagonist', 'Antiflatulent', 'Antiseptic', 'Antitussive', 'Laxative',
    'Opioid Agonist-Antagonist', 'Radiology', 'Respiratory', 'Smoking cessation', 'Vaccine'
]

# Crear diccionario con los valores únicos de cada uno
valores_unicos = {}
for med in medicamentos_consulta:
    if med in df_variables_nuevas.columns:
        valores_unicos[med] = sorted(df_variables_nuevas[med].dropna().unique())

# Convertir a DataFrame para mostrar
df_valores_unicos = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in valores_unicos.items()]))
df_valores_unicos 

In [ ]:
for var in medicamentos_consulta:
    nueva_col = f'toma_{var.lower().replace(" ", "_")}'
    df_variables_nuevas[nueva_col] = df_variables_nuevas[var].notnull() & (df_variables_nuevas[var] > 0)
    df_variables_nuevas[nueva_col] = df_variables_nuevas[nueva_col].astype(int)


In [ ]:
 df_variables_nuevas.head()

In [ ]:
df_variables_nuevas.shape[0]

In [ ]:
import pandas as pd
import numpy as np
# Copiar DataFrame
df_categorized = df_variables_nuevas.copy()


# Lista de columnas a clasificar
columns = [
    'Analgesic', 'Antibiotic', 'Anticoagulant', 'Antidepressant',
    'Antihypertensive', 'Antiplatelet',
    'Electrolyte', 'Fluid & Electrolyte', 'Lipid-lowering', 'Endocrine & Metabolic'

]

# Función para clasificar la dosis según los percentiles
def clasificar_dosis(col):
    percentil_25 = col.quantile(0.25)
    percentil_75 = col.quantile(0.75)
    return col.apply(lambda x: 0 if pd.isnull(x)
                     else 1 if x < percentil_25
                     else 2 if x <= percentil_75
                     else 3)

# Crear nuevas columnas clasificadas
for col in columns:
    if col in df_categorized.columns:
        df_categorized[col + '_class'] = clasificar_dosis(df_categorized[col])

In [ ]:
df_categorized.head()

In [ ]:
for col in columns:
    class_col = col + '_class'
    if class_col in df_categorized.columns:
        print(f"Distribución para {col}:")
        print(df_categorized[class_col].value_counts().sort_index())
        print("\n")

In [ ]:
df_categorized.to_excel(r'DATA_ROOT\MODELO\dataset_completo.xlsx', index=True)

In [ ]:
df_modelo=df_categorized.copy()

### PREPARACIÓN PARA EL MODELO PREDICTIVO EN R:

####  Definir target y features

In [ ]:
# # Lista de columnas que serán utilizadas como features
# features = [
#     'gender', 'anchor_age', 'Cardiología', 'Cirugía / Procedimientos Médicos', 'Enfermedades Infecciosas',
#     'Factores Sociales / Exposición / Historial', 'Gastroenterología / Digestivo', 'Insuficiencia Renal',
#     'Metabólicas Y Nutricionales', 'Nefrología', 'Neumología', 'Oncología', 'Reumatología',
#     'Salud Mental Y Neurológica', 'Analgesic', 'Antagonist', 'Antibiotic', 'Anticoagulant',
#     'Antidepressant', 'Antiflatulent', 'Antihypertensive', 'Antiplatelet', 'Antiseptic', 'Antitussive',
#     'Electrolyte', 'Endocrine & Metabolic', 'Fluid & Electrolyte', 'Laxative', 'Lipid-lowering',
#     'Opioid Agonist-Antagonist', 'Radiology', 'Respiratory', 'Smoking cessation', 'Vaccine',
#     'Anion Gap-Blood-Chemistry', 'Bicarbonate-Blood-Chemistry', 'Calcium, Total-Blood-Chemistry',
#     'Chloride-Blood-Chemistry', 'Glucose-Blood-Chemistry', 'Hematocrit-Blood-Hematology',
#     'Hemoglobin-Blood-Hematology', 'MCH-Blood-Hematology', 'MCHC-Blood-Hematology',
#     'MCV-Blood-Hematology', 'Magnesium-Blood-Chemistry', 'Phosphate-Blood-Chemistry',
#     'Platelet Count-Blood-Hematology', 'Potassium-Blood-Chemistry', 'RDW-Blood-Hematology',
#     'Red Blood Cells-Blood-Hematology', 'Sodium-Blood-Chemistry', 'White Blood Cells-Blood-Hematology',
#     'murio', 'toma_iron_supplements', 'toma_cardiovascular', 'toma_renal', 'toma_corticosteroid',
#     'toma_diuretic', 'toma_vasodilator', 'toma_vasopressor', 'toma_anesthetic', 'toma_supplement',
#     'toma_antipsychotic', 'toma_proton_pump_inhibitor', 'toma_urological', 'toma_immunosuppressant',
#     'toma_sedative', 'toma_antihistamine', 'toma_h2_antihistamine', 'eGFR', 'BUN/Creatinina', 
# ]

# # Definimos X (features) y y (target)
# X = df_modelo[features]
# y = df_modelo['Insuficiencia Renal']  # Reemplaza 'target' por el nombre real de tu columna objetivo si es distinto


In [ ]:
# Definir las features del modelo predictivo
features = [
    'gender', 'anchor_age', 'Cardiología', 'Cirugía / Procedimientos Médicos', 'Enfermedades Infecciosas',
    'Factores Sociales / Exposición / Historial', 'Gastroenterología / Digestivo', 'Insuficiencia Renal',
    'Metabólicas Y Nutricionales', 'Nefrología', 'Neumología', 'Oncología', 'Reumatología',
    'Salud Mental Y Neurológica', 'Analgesic', 'Antagonist', 'Antibiotic', 'Anticoagulant',
    'Antidepressant', 'Antiflatulent', 'Antihypertensive', 'Antiplatelet', 'Antiseptic', 'Antitussive',
    'Electrolyte', 'Fluid & Electrolyte', 'Laxative', 'Lipid-lowering',
    'Opioid Agonist-Antagonist', 'Radiology', 'Respiratory', 'Smoking cessation', 'Vaccine',
    'Anion Gap-Blood-Chemistry', 'Bicarbonate-Blood-Chemistry', 'Calcium, Total-Blood-Chemistry',
    'Chloride-Blood-Chemistry', 'Glucose-Blood-Chemistry', 'Hematocrit-Blood-Hematology',
    'Hemoglobin-Blood-Hematology', 'MCH-Blood-Hematology', 'MCHC-Blood-Hematology',
    'MCV-Blood-Hematology', 'Magnesium-Blood-Chemistry', 'Phosphate-Blood-Chemistry',
    'Platelet Count-Blood-Hematology', 'Potassium-Blood-Chemistry', 'RDW-Blood-Hematology',
    'Red Blood Cells-Blood-Hematology', 'Sodium-Blood-Chemistry', 'White Blood Cells-Blood-Hematology',
    'murio', 'toma_iron_supplements', 'toma_cardiovascular', 'toma_renal', 'toma_corticosteroid',
    'toma_diuretic', 'toma_vasodilator', 'toma_vasopressor', 'toma_anesthetic', 'toma_supplement',
    'toma_antipsychotic', 'toma_proton_pump_inhibitor', 'toma_urological', 'toma_immunosuppressant',
    'toma_sedative', 'toma_antihistamine', 'toma_h2_antihistamine', 'eGFR', 'BUN/Creatinina', 
    'Analgesic_class', 'Antibiotic_class', 'Anticoagulant_class', 'Antidepressant_class', 
    'Antihypertensive_class', 'Antiplatelet_class',
    'Electrolyte_class', 'Fluid & Electrolyte_class', 'Lipid-lowering_class', 'Endocrine & Metabolic_class'
]

# Asegurarse de que todas las columnas están en el DataFrame
df_modelo = df_categorized[features]

# Verificar las primeras filas para confirmar que las features están bien seleccionadas
print(df_categorized.head())


# Definimos X (features) y y (target)
X = df_modelo[features]
y = df_modelo['Insuficiencia Renal']  # Reemplaza 'target' por el nombre real de tu columna objetivo si es distinto



In [ ]:
X

In [ ]:
# Obtener los tipos de datos de las columnas
data_types = X.dtypes

# Clasificar las columnas por tipo de dato
types_dict = {}
for col, dtype in data_types.items():
    if dtype not in types_dict:
        types_dict[dtype] = [col]
    else:
        types_dict[dtype].append(col)

# Mostrar las columnas clasificadas por tipo de dato
for dtype, columns in types_dict.items():
    print(f"{dtype}: {columns}")


In [ ]:
#Transformamos los booleanos a os y 1s: 
# Asegúrate de que las columnas booleans estén correctamente identificadas
boolean_columns = ['Cardiología', 'Cirugía / Procedimientos Médicos', 'Enfermedades Infecciosas', 
                   'Factores Sociales / Exposición / Historial', 'Gastroenterología / Digestivo', 
                   'Insuficiencia Renal', 'Metabólicas Y Nutricionales', 'Nefrología', 'Neumología', 
                   'Oncología', 'Reumatología', 'Salud Mental Y Neurológica', 'toma_iron_supplements', 
                   'toma_cardiovascular', 'toma_renal', 'toma_corticosteroid', 'toma_diuretic', 
                   'toma_vasodilator', 'toma_vasopressor', 'toma_anesthetic', 'toma_supplement', 
                   'toma_antipsychotic', 'toma_proton_pump_inhibitor', 'toma_urological', 'toma_immunosuppressant', 
                   'toma_sedative', 'toma_antihistamine', 'toma_h2_antihistamine']

# Convertir las columnas booleanas a 0 y 1
X[boolean_columns] = X[boolean_columns].astype(int)

# Verificar los primeros registros para asegurarse de que la conversión se haya realizado correctamente
print(df_variables_nuevas[boolean_columns].head())


In [ ]:
X.to_excel(r'DATA_ROOT\MODELO\dataset_features.xlsx', index=True)